# Figure — R² Ablation: Subject Size

Line chart comparing R² (train & test) across nine subject-count levels
(25 → 1,293 training subjects) for the **Generalisable** (GS config)
and **Fit-Biased** (stress-test config) models.

Data source (both curves, both configs):
- `ABBLATION_RESULTS_PAPER / Subject_Size_Uncertainty / {Generalisable,FitBiased} / Subject_Size_{n} / TrainDraws_Distribution.csv`

**Training** and **Testing** = mean ± 1 SD across the same 10 independent training draws
per level (repeat-based uncertainty band; single deterministic draw at N=1,293, so no
band there). Each draw yields one Train R² and one Test R², so both curves' bars come
from that same set of draws — the Test band reflects how much a model's test-set
performance varies due to which subjects were drawn for training.

One figure per BP target. Set `TARGET` below to switch between SBP / DBP / MAP.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path("../").resolve()))

from fig_style import (
    DPI, W_FULL, W_SINGLE, ASPECT, FONT_FAMILY,
    MIN_PX_FULL, MIN_PX_SINGLE,
    apply_base_style, save_fig,
)
from local_paths import ABBLATION_RESULTS_PAPER, FIGURES_PAPER

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

FIG_OUT = FIGURES_PAPER
FIG_OUT.mkdir(parents=True, exist_ok=True)
print(f"Output directory : {FIG_OUT.resolve()}")
print(f"Full-page  : {W_FULL:.2f} × {W_FULL*ASPECT:.2f} in  →  "
      f"{round(W_FULL*DPI)} × {round(W_FULL*ASPECT*DPI)} px @ {DPI} dpi")
print(f"Single-col : {W_SINGLE:.2f} × {W_SINGLE*ASPECT:.2f} in  →  "
      f"{round(W_SINGLE*DPI)} × {round(W_SINGLE*ASPECT*DPI)} px @ {DPI} dpi")

## 1 · Load data

In [ ]:
# ── Change TARGET here to produce DBP or MAP figures ─────────────────────────
TARGET     = "MAP"

N_SUBJECTS = [25, 50, 100, 200, 400, 600, 800, 1000, 1293]
X_LABELS   = [str(n) for n in N_SUBJECTS]

UNCERTAINTY_BASE = ABBLATION_RESULTS_PAPER / "Subject_Size_Uncertainty"


def level_dir(variant: str, n_subjects: int) -> Path:
    return UNCERTAINTY_BASE / variant / f"Subject_Size_{n_subjects}"


def load_level(variant: str, n_subjects: int) -> pd.DataFrame:
    return pd.read_csv(level_dir(variant, n_subjects) / "TrainDraws_Distribution.csv")


for variant in ("Generalisable", "FitBiased"):
    missing = [n for n in N_SUBJECTS if not level_dir(variant, n).exists()]
    print(f"{variant:14s}: {len(N_SUBJECTS) - len(missing)}/{len(N_SUBJECTS)} levels found"
          + (f"  MISSING: {missing}" if missing else ""))

## 2 · Plot

In [ ]:
# ── Palette (consistent with Fig 6 / Sample Size figure) ─────────────────────
COLOR_GEN   = "#E84855"   # Watermelon → Generalisable
COLOR_FB    = "#27AE60"   # Green      → Fit-Biased
COLOR_LABEL = "#333333"   # dark gray for all data labels
COLOR_ERR   = "black"     # error bars — drawn on top of markers, always visible
LW          = 1.8
MS          = 5
LABEL_X_OFF = 0.15  # small horizontal nudge for label text only (Generalisable
                     # left, Fit-Biased right), purely cosmetic so red/green
                     # text doesn't sit on the same x column — independent of
                     # where the bar itself is drawn
TEST_BAR_DODGE  = 0.036  # horizontal nudge for the Test error bars only, and
                          # only at the two levels below — reduced ~60% from
                          # an earlier across-the-board version that dodged
                          # every point and every curve, which was too much;
                          # Train bars never overlapped so they stay un-dodged
TEST_BAR_DODGE_LEVELS = {25, 50}  # N values where the Test bars are close
                                   # enough to collide; everywhere else the
                                   # bars sit at the plain tick like Train does
Y_LIM       = (-34, 115)  # bottom extended well past -22: the worst whisker
                           # across all three targets at N=25 is FitBiased
                           # DBP Test (mean -17.17, SD 11.13 -> whisker tip
                           # -28.30), which the old -22 limit was clipping
                           # outright. -34 keeps that fully visible with room
                           # for its label below the cap, for any TARGET.
LABEL_Y_MARGIN = 3  # keep data labels at least this far from the y-axis edges,
                     # so a large error bar near the bottom/top of the range
                     # doesn't push its label into the axis tick text


def get_band(variant, target, x_vals, subset):
    """Mean R² (%) ± 1 SD across independent training draws, per level, for
    the given subset ('Train' or 'Test') — both curves use the identical
    repeat set, so both get the same treatment. Zero-variance levels
    (single deterministic draw, e.g. N=1,293) get 0 SD."""
    means, stds = [], []
    for x in x_vals:
        df = load_level(variant, x)
        vals = df.loc[(df["target"] == target) & (df["subset"] == subset), "R2"] * 100
        means.append(vals.mean())
        stds.append(vals.std(ddof=1) if len(vals) > 1 else 0.0)
    return np.array(means), np.array(stds)


def _mask_zero_err(err):
    """NaN out exactly-zero SD entries so matplotlib draws no bar/cap at all
    for deterministic (single-draw) levels, instead of a zero-height tick."""
    return np.where(err == 0, np.nan, err)


def _clip_label_y(y):
    return float(np.clip(y, Y_LIM[0] + LABEL_Y_MARGIN, Y_LIM[1] - LABEL_Y_MARGIN))


def _add_labels(ax, xi, vals, va, y_off, x_off, fs):
    for i, v in enumerate(vals):
        if not np.isnan(v):
            ax.text(
                xi[i] + x_off, _clip_label_y(v + y_off),
                f"{v:.2f}%",
                ha="center", va=va,
                fontsize=fs, fontfamily=FONT_FAMILY,
                color=COLOR_LABEL,
            )


def _add_paired_labels(ax, xi, vals_a, err_a, x_off_a, vals_b, err_b, x_off_b, y_off, fs):
    """Label two curves that converge/cross (e.g. gen_te vs fb_te): at each
    point, the higher value's label goes above and the lower value's label
    goes below, decided per-point rather than by a fixed index range — a
    fixed range only works for one specific grid size and breaks as soon as
    the number of levels changes. Padding is y_off PLUS that point's own
    error-bar size (err_a/err_b, raw — not the NaN-masked version — so 0 at
    deterministic levels just falls back to y_off) so the label clears its
    own whisker cap instead of overlapping it when the SD is large — then
    clipped to stay LABEL_Y_MARGIN away from the axis edges, since a huge
    whisker near the bottom/top of the range would otherwise push the label
    into the axis tick text instead of just past its own cap."""
    for i in range(len(xi)):
        a, b = vals_a[i], vals_b[i]
        if np.isnan(a) or np.isnan(b):
            continue
        va_a, va_b = ("bottom", "top") if a >= b else ("top", "bottom")
        pad_a = y_off + (err_a[i] if not np.isnan(err_a[i]) else 0.0)
        pad_b = y_off + (err_b[i] if not np.isnan(err_b[i]) else 0.0)
        ax.text(xi[i] + x_off_a, _clip_label_y(a + (pad_a if va_a == "bottom" else -pad_a)),
                f"{a:.2f}%", ha="center", va=va_a, fontsize=fs,
                fontfamily=FONT_FAMILY, color=COLOR_LABEL)
        ax.text(xi[i] + x_off_b, _clip_label_y(b + (pad_b if va_b == "bottom" else -pad_b)),
                f"{b:.2f}%", ha="center", va=va_b, fontsize=fs,
                fontfamily=FONT_FAMILY, color=COLOR_LABEL)


def make_subject_size_fig(width_in: float, target: str = TARGET):
    height_in = width_in * ASPECT
    is_small  = width_in < 5
    label_fs  = 5.5 if is_small else 7.0
    tick_fs   = 6   if is_small else 8
    title_fs  = 7   if is_small else 10
    leg_fs    = 6   if is_small else 8
    note_fs   = 4.5 if is_small else 6.0
    lw        = 1.2 if is_small else LW
    ms        = 3.5 if is_small else MS

    xi = np.arange(len(N_SUBJECTS))
    test_dodge = np.array([TEST_BAR_DODGE if n in TEST_BAR_DODGE_LEVELS else 0.0
                            for n in N_SUBJECTS])

    fig, ax = plt.subplots(
        figsize=(width_in, height_in),
        dpi=DPI,
        layout="constrained",
    )

    gen_tr, gen_tr_err = get_band("Generalisable", target, N_SUBJECTS, "Train")
    gen_te, gen_te_err = get_band("Generalisable", target, N_SUBJECTS, "Test")
    fb_tr,  fb_tr_err  = get_band("FitBiased", target, N_SUBJECTS, "Train")
    fb_te,  fb_te_err  = get_band("FitBiased", target, N_SUBJECTS, "Test")

    # Reference line at y=0
    ax.axhline(0, color="#888888", lw=0.9, ls="-", zorder=2)

    # Both Train and Test: mean line + colored marker (zorder=3) stay exactly
    # on the tick, with the ±1 SD error bar (across the same 10 repeat draws)
    # drawn separately in black on top (zorder=4) — a bar is always visible
    # as a small black tick even when its span is smaller than the marker
    # itself, instead of disappearing under it. Exactly-zero SD (deterministic
    # single-draw levels) is masked to NaN so no bar/cap renders there at
    # all. Train bars sit at the plain tick (never overlapped); Test bars
    # are nudged ±TEST_BAR_DODGE off the tick only at N=25/50, where the two
    # configs' values are close enough for the bars to collide otherwise —
    # everywhere else Test bars sit at the plain tick too.
    ax.plot(xi, gen_tr, color=COLOR_GEN, ls="-", marker="o", lw=lw, ms=ms, zorder=3)
    ax.errorbar(xi, gen_tr, yerr=_mask_zero_err(gen_tr_err), fmt="none", ecolor=COLOR_ERR,
                capsize=3, capthick=1, elinewidth=1.1, zorder=4)
    ax.plot(xi, gen_te, color=COLOR_GEN, ls="--", marker="o", lw=lw, ms=ms, zorder=3,
            markerfacecolor="white", markeredgewidth=1.2)
    ax.errorbar(xi - test_dodge, gen_te, yerr=_mask_zero_err(gen_te_err), fmt="none", ecolor=COLOR_ERR,
                capsize=3, capthick=1, elinewidth=1.1, zorder=4)
    ax.plot(xi, fb_tr, color=COLOR_FB, ls="-", marker="s", lw=lw, ms=ms, zorder=3)
    ax.errorbar(xi, fb_tr, yerr=_mask_zero_err(fb_tr_err), fmt="none", ecolor=COLOR_ERR,
                capsize=3, capthick=1, elinewidth=1.1, zorder=4)
    ax.plot(xi, fb_te, color=COLOR_FB, ls="--", marker="s", lw=lw, ms=ms, zorder=3,
            markerfacecolor="white", markeredgewidth=1.2)
    ax.errorbar(xi + test_dodge, fb_te, yerr=_mask_zero_err(fb_te_err), fmt="none", ecolor=COLOR_ERR,
                capsize=3, capthick=1, elinewidth=1.1, zorder=4)

    # Data labels: Train curves are well-separated from everything, always above.
    # Test curves converge/cross, so pick above/below per-point based on which
    # is higher at that x — avoids the overlap a fixed top/bottom split
    # produces once they're this close together.
    _add_labels(ax, xi, gen_tr, "bottom", +2.5, -LABEL_X_OFF, label_fs)
    _add_labels(ax, xi, fb_tr,  "bottom", +2.5, +LABEL_X_OFF, label_fs)
    _add_paired_labels(ax, xi, gen_te, gen_te_err, -LABEL_X_OFF, fb_te, fb_te_err, +LABEL_X_OFF, 2.5, label_fs)

    # Axes
    ax.set_xticks(xi)
    ax.set_xticklabels(X_LABELS, fontsize=tick_fs, fontfamily=FONT_FAMILY)
    ax.set_xlim(-0.6, len(N_SUBJECTS) - 0.4)
    ax.set_ylim(*Y_LIM)
    ax.yaxis.set_major_locator(mticker.MultipleLocator(20))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))
    ax.set_xlabel(
        "Number of Training Subjects",
        fontsize=tick_fs + 1, fontfamily=FONT_FAMILY,
    )
    ax.set_ylabel("R²", fontsize=tick_fs + 1, fontfamily=FONT_FAMILY)

    apply_base_style(ax, grid_axis="y")
    ax.tick_params(axis="both", labelsize=tick_fs)

    # Legend: filled marker = Training, hollow marker = Testing
    legend_elements = [
        Line2D([0],[0], color=COLOR_GEN, ls="-",  marker="o", lw=LW, ms=MS,
               label="Training — Generalisable"),
        Line2D([0],[0], color=COLOR_GEN, ls="--", marker="o", lw=LW, ms=MS,
               markerfacecolor="white", markeredgewidth=1.2,
               label="Testing — Generalisable"),
        Line2D([0],[0], color=COLOR_FB,  ls="-",  marker="s", lw=LW, ms=MS,
               label="Training — Fit-Biased"),
        Line2D([0],[0], color=COLOR_FB,  ls="--", marker="s", lw=LW, ms=MS,
               markerfacecolor="white", markeredgewidth=1.2,
               label="Testing — Fit-Biased"),
    ]
    ax.legend(
        handles=legend_elements,
        loc="upper right", ncol=1,
        fontsize=leg_fs,
        handlelength=2.5,
        frameon=True, framealpha=0.9, edgecolor="#CCCCCC",
    )

    ax.text(
        0.99, 0.02,
        "Bars (black): ±1 SD across the same 10 draws for Train and Test (none at N=1,293).\n"
        "Test bars nudged left/right of the tick at N=25/50 only, where they'd otherwise overlap.",
        transform=ax.transAxes, ha="right", va="bottom",
        fontsize=note_fs, fontfamily=FONT_FAMILY, color="#888888",
    )

    ax.set_title(
        f"R² Performance Across Different Numbers of Training Subjects ({target})\n"
        "Generalisable vs Fit-Biased Model Configurations",
        fontsize=title_fs + 1,
        fontfamily=FONT_FAMILY,
    )
    return fig


fig_full   = make_subject_size_fig(W_FULL)
fig_single = make_subject_size_fig(W_SINGLE)

save_fig(fig_full,   f"Fig_Ablation_SubjectSize_{TARGET}_full",   FIG_OUT)
save_fig(fig_single, f"Fig_Ablation_SubjectSize_{TARGET}_single", FIG_OUT)

print(f"Full-page  : {round(W_FULL*DPI)} × {round(W_FULL*ASPECT*DPI)} px")
print(f"Single-col : {round(W_SINGLE*DPI)} × {round(W_SINGLE*ASPECT*DPI)} px")
plt.show()

## 3 · Verify pixel counts

In [ ]:
from PIL import Image

for fname, req_w in [
    (f"Fig_Ablation_SubjectSize_{TARGET}_full.png",   MIN_PX_FULL),
    (f"Fig_Ablation_SubjectSize_{TARGET}_single.png", MIN_PX_SINGLE),
]:
    with Image.open(FIG_OUT / fname) as im:
        w, h = im.size
    ok = "\u2705" if w >= req_w else "\u274c"
    print(f"{ok} {fname}: {w} \u00d7 {h} px  (min required: {req_w} px wide)")